Allergy_ver2와 식품데이터_최종_표준화_ver3 파일을 이용해서 알레르기 태깅이 마무리된 엑셀 파일을 출력하는 것이 목표

In [1]:
import json
import pandas as pd

def analyze_allergens_and_export(food_data_path, allergy_info_path, output_excel_path):
    """
    식품 데이터 JSON 파일을 읽어 알레르기 정보를 태깅하고 엑셀 파일로 저장합니다.

    Args:
        food_data_path (str): 분석할 식품 데이터 JSON 파일 경로
        allergy_info_path (str): 알레르기 규칙 JSON 파일 경로
        output_excel_path (str): 저장할 엑셀 파일 경로
    """
    # --- 1. 파일 로드 ---
    try:
        with open(food_data_path, 'r', encoding='utf-8') as f:
            food_data = json.load(f)
        with open(allergy_info_path, 'r', encoding='utf-8') as f:
            allergy_info = json.load(f)
        print("✅ JSON 파일 로드 완료")
    except FileNotFoundError as e:
        print(f"오류: 파일을 찾을 수 없습니다. ({e.filename})")
        return
    except json.JSONDecodeError:
        print(f"오류: JSON 파일 형식이 올바르지 않습니다.")
        return

    # --- 2. 알레르기 분석 로직 ---
    print("⏳ 알레르기 정보 분석 중...")
    
    # 각 제품을 순회하며 분석
    for product in food_data:
        detected_allergens = set()
        all_ingredients = product.get("원재료명", [])

        if not all_ingredients:
            product["알레르기"] = []
            continue

        # 각 알레르기 카테고리에 대해 검사
        for category, rules in allergy_info.items():
            is_category_tagged = False
            
            # 0단계: 예외 키워드와 정확히 일치하는 원재료 필터링
            exact_exclusions = set(rules.get("exclusions", []))
            ingredients_to_check = [ing for ing in all_ingredients if ing not in exact_exclusions]

            if not ingredients_to_check:
                continue

            # 1단계: 키워드 완전 일치 검사
            for ingredient in ingredients_to_check:
                if ingredient in rules.get("keyword", []):
                    detected_allergens.add(category)
                    is_category_tagged = True
                    break
            if is_category_tagged: continue

            # 2단계: 키워드 포함 및 복합 규칙 검사
            for ingredient in ingredients_to_check:
                matching_keywords = [k for k in rules.get("keyword", []) if k in ingredient]
                if not matching_keywords: continue

                matching_exclusions = [e for e in rules.get("exclusions", []) if e in ingredient]
                
                longest_k = max(matching_keywords, key=len)
                longest_e = max(matching_exclusions, key=len) if matching_exclusions else ""

                if len(longest_k) > len(longest_e):
                    # 접두사 예외 처리
                    if any(ingredient.startswith(prefix) for prefix in rules.get('exclusions_first_word', [])):
                        continue
                    
                    # 특수 규칙 검사
                    is_special = longest_k in rules.get("only_one", []) + rules.get("last_word", []) + rules.get("first_word", [])
                    tagged_by_special_rule = False
                    if is_special:
                        if (longest_k in rules.get("only_one", []) and ingredient == longest_k) or \
                           (longest_k in rules.get("last_word", []) and ingredient.endswith(longest_k)) or \
                           (longest_k in rules.get("first_word", []) and ingredient.startswith(longest_k)):
                            tagged_by_special_rule = True
                    
                    if not is_special or tagged_by_special_rule:
                        detected_allergens.add(category)
                        is_category_tagged = True
                        break
            if is_category_tagged: continue
        
        # '알레르기' 키에 분석 결과(정렬된 리스트)를 추가
        product["알레르기"] = sorted(list(detected_allergens))

    print("✅ 알레르기 분석 완료")

    # --- 3. 엑셀 파일로 내보내기 ---
    try:
        print(f"⏳ 결과를 '{output_excel_path}' 파일로 저장 중...")
        # 분석 결과를 pandas DataFrame으로 변환
        df = pd.DataFrame(food_data)
        
        # '원재료명'과 '알레르기' 리스트를 보기 좋게 문자열로 변환 (선택 사항)
        df['원재료명'] = df['원재료명'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)
        df['알레르기'] = df['알레르기'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)

        # DataFrame을 엑셀 파일로 저장 (인덱스는 저장하지 않음)
        df.to_excel(output_excel_path, index=False, engine='openpyxl')
        print(f"🎉 성공! '{output_excel_path}' 파일이 생성되었습니다.")
    except Exception as e:
        print(f"오류: 엑셀 파일 저장에 실패했습니다. - {e}")

# --- 메인 실행 블록 ---
if __name__ == '__main__':
    # 설정할 파일 경로
    FOOD_DATA_FILE = '식품데이터_최종_표준화_ver3.json'
    ALLERGY_RULES_FILE = '../Data_Storage/표준화 데이터/Allergy_ver_2.json'
    OUTPUT_EXCEL_FILE = '알레르기_분류_결과.xlsx'
    
    # 함수 실행
    analyze_allergens_and_export(FOOD_DATA_FILE, ALLERGY_RULES_FILE, OUTPUT_EXCEL_FILE)

✅ JSON 파일 로드 완료
⏳ 알레르기 정보 분석 중...
✅ 알레르기 분석 완료
⏳ 결과를 '알레르기_분류_결과.xlsx' 파일로 저장 중...
🎉 성공! '알레르기_분류_결과.xlsx' 파일이 생성되었습니다.


# 데이터 추가 정제

## 삭제 해야 하는것(원재료)
- 빈칸
- 소스(단독)
- 향료(단독)
- 기타 가공품(단독)
- 기타가공품(단독)
- 복합조미식품(단독)
- 과채가공품(단독)
- 두류가공품(단독)
- 과채주스(단독)

## 삭제해야 하는 카테고리
- 감색소
- 계피알데히드
- 고령자용 영양조제식품
- 과당
- 기타천연첨가물
- 기타혼합제제



In [1]:
import json

def refine_and_delete_food_data(input_path, output_path):
    """
    식품 데이터 JSON 파일을 두 가지 삭제 규칙에 따라 정제하고 새로운 파일로 저장합니다.
    - 특정 카테고리에 해당하는 제품 삭제
    - 특정 원재료명만 단독으로 포함하는 제품 삭제
    """
    # --- 1. 정제 규칙 정의 ---

    # 규칙 1: 원재료명 리스트에 단독으로 있을 때, 제품 전체를 삭제할 단어 목록
    single_ingredients_to_delete = {
        "소스", "향료", "기타 가공품", "기타가공품", 
        "복합조미식품", "과채가공품", "두류가공품", "과채주스"
    }

    # 규칙 2: 이 카테고리에 해당하는 데이터는 전체 삭제
    categories_to_delete = {
        "감색소", "계피알데히드", "고령자용 영양조제식품", 
        "과당", "기타천연첨가물", "기타혼합제제"
    }
    
    # --- 2. 데이터 로드 ---
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            food_data = json.load(f)
        print(f"✅ 원본 파일 로드 완료: 총 {len(food_data)}개 제품")
    except FileNotFoundError:
        print(f"오류: '{input_path}' 파일을 찾을 수 없습니다.")
        return
    except json.JSONDecodeError:
        print(f"오류: '{input_path}' 파일이 올바른 JSON 형식이 아닙니다.")
        return

    # --- 3. 데이터 정제 작업 ---
    refined_data = []
    deleted_by_category_count = 0
    deleted_by_ingredient_count = 0

    # 전체 데이터를 하나씩 순회
    for product in food_data:
        # 규칙 2: 카테고리 필터링 (첫 번째 삭제 조건)
        if product.get("카테고리") in categories_to_delete:
            deleted_by_category_count += 1
            continue  # 제품 삭제

        # 원재료 리스트에서 '빈칸' (공백 문자열) 먼저 제거
        ingredients = product.get("원재료명", [])
        cleaned_ingredients = [ing for ing in ingredients if ing and ing.strip()]
        
        # 규칙 1: 원재료명 필터링 (두 번째 삭제 조건)
        if len(cleaned_ingredients) == 1 and cleaned_ingredients[0] in single_ingredients_to_delete:
            deleted_by_ingredient_count += 1
            continue  # 제품 삭제

        # 어떤 삭제 규칙에도 해당하지 않는 제품만 새 리스트에 추가
        # (이때, 빈칸이 제거된 깔끔한 원재료명으로 업데이트하여 저장)
        product["원재료명"] = cleaned_ingredients
        refined_data.append(product)

    print(f"⏳ 정제 작업 완료...")

    # --- 4. 결과 저장 ---
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(refined_data, f, ensure_ascii=False, indent=4)
        
        print("\n--- 📊 정제 결과 요약 ---")
        print(f"카테고리 규칙으로 삭제된 제품 수: {deleted_by_category_count}개")
        print(f"단독 원재료 규칙으로 삭제된 제품 수: {deleted_by_ingredient_count}개")
        print(f"총 삭제된 제품 수: {deleted_by_category_count + deleted_by_ingredient_count}개")
        print(f"최종 저장된 제품 수: {len(refined_data)}개")
        print(f"🎉 성공! '{output_path}' 파일이 생성되었습니다.")

    except Exception as e:
        print(f"오류: 결과를 파일로 저장하는 데 실패했습니다 - {e}")


# --- 메인 실행 블록 ---
if __name__ == '__main__':
    # 입력 파일과 출력 파일 이름 설정
    INPUT_FILE = '식품데이터_최종_표준화_ver3.json'
    OUTPUT_FILE = '식품데이터_최종_표준화_ver4.json'
    
    refine_and_delete_food_data(INPUT_FILE, OUTPUT_FILE)

✅ 원본 파일 로드 완료: 총 91675개 제품
⏳ 정제 작업 완료...

--- 📊 정제 결과 요약 ---
카테고리 규칙으로 삭제된 제품 수: 12개
단독 원재료 규칙으로 삭제된 제품 수: 377개
총 삭제된 제품 수: 389개
최종 저장된 제품 수: 91286개
🎉 성공! '식품데이터_최종_표준화_ver4.json' 파일이 생성되었습니다.


In [2]:
import json

def separate_food_data(input_path, output_kept_path, output_removed_path):
    """
    식품 데이터를 규칙에 따라 정제된 데이터와 삭제된 데이터로 분리하여
    각각 별도의 JSON 파일로 저장합니다.
    """
    # --- 1. 정제 규칙 정의 ---

    # 규칙 1: 제품을 삭제할 단독 원재료 목록
    single_ingredients_to_delete = {
        "소스", "향료", "기타 가공품", "기타가공품", 
        "복합조미식품", "과채가공품", "두류가공품", "과채주스"
    }

    # 규칙 2: 제품을 삭제할 카테고리 목록
    categories_to_delete = {
        "감색소", "계피알데히드", "고령자용 영양조제식품", 
        "과당", "기타천연첨가물", "기타혼합제제"
    }
    
    # --- 2. 데이터 로드 ---
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            food_data = json.load(f)
        print(f"✅ 원본 파일 로드 완료: 총 {len(food_data)}개 제품")
    except FileNotFoundError:
        print(f"오류: '{input_path}' 파일을 찾을 수 없습니다.")
        return
    except json.JSONDecodeError:
        print(f"오류: '{input_path}' 파일이 올바른 JSON 형식이 아닙니다.")
        return

    # --- 3. 데이터 분리 작업 ---
    kept_data = []      # 살아남은 데이터를 담을 리스트
    removed_data = []   # 삭제된 데이터를 담을 리스트

    # 전체 데이터를 하나씩 순회
    for product in food_data:
        reason_for_removal = None  # 삭제 사유 초기화

        # 규칙 2: 카테고리 필터링
        category = product.get("카테고리")
        if category in categories_to_delete:
            reason_for_removal = f"카테고리 필터링: '{category}'"

        # 카테고리에서 이미 걸러지지 않은 경우에만 원재료 규칙 검사
        if not reason_for_removal:
            ingredients = product.get("원재료명", [])
            # 원재료 리스트에서 '빈칸' (공백 문자열) 제거
            cleaned_ingredients = [ing for ing in ingredients if ing and ing.strip()]
            
            # 규칙 1: 단독 원재료 필터링
            if len(cleaned_ingredients) == 1 and cleaned_ingredients[0] in single_ingredients_to_delete:
                reason_for_removal = f"단독 원재료 필터링: '{cleaned_ingredients[0]}'"

        # --- 최종 분리 ---
        if reason_for_removal:
            # 삭제 사유가 있다면, 삭제 리스트로 이동
            product["삭제사유"] = reason_for_removal  # 삭제 이유를 제품 정보에 추가
            removed_data.append(product)
        else:
            # 삭제 사유가 없다면, 유지 리스트로 이동
            # (이때, 빈칸이 제거된 깔끔한 원재료명으로 업데이트)
            product["원재료명"] = cleaned_ingredients
            kept_data.append(product)

    print(f"⏳ 데이터 분리 작업 완료...")

    # --- 4. 결과 파일들 저장 ---
    try:
        # 4-a: 정제된 데이터 저장
        with open(output_kept_path, 'w', encoding='utf-8') as f:
            json.dump(kept_data, f, ensure_ascii=False, indent=4)
        
        # 4-b: 삭제된 데이터 저장
        with open(output_removed_path, 'w', encoding='utf-8') as f:
            json.dump(removed_data, f, ensure_ascii=False, indent=4)
        
        print("\n--- 📊 정제 결과 요약 ---")
        print(f"최종 저장된 제품 수: {len(kept_data)}개")
        print(f"삭제되어 별도 저장된 제품 수: {len(removed_data)}개")
        print(f"🎉 성공! 2개의 파일이 생성되었습니다:")
        print(f"  - 정제된 데이터: '{output_kept_path}'")
        print(f"  - 삭제된 데이터: '{output_removed_path}'")

    except Exception as e:
        print(f"오류: 결과를 파일로 저장하는 데 실패했습니다 - {e}")


# --- 메인 실행 블록 ---
if __name__ == '__main__':
    # 입력 파일과 2개의 출력 파일 이름 설정
    INPUT_FILE = '식품데이터_최종_표준화_ver3.json'
    OUTPUT_KEPT_FILE = '식품데이터_최종_정제_ver4.json'
    OUTPUT_REMOVED_FILE = '식품데이터_삭제됨_ver4.json'
    
    separate_food_data(INPUT_FILE, OUTPUT_KEPT_FILE, OUTPUT_REMOVED_FILE)

✅ 원본 파일 로드 완료: 총 91675개 제품
⏳ 데이터 분리 작업 완료...

--- 📊 정제 결과 요약 ---
최종 저장된 제품 수: 91286개
삭제되어 별도 저장된 제품 수: 389개
🎉 성공! 2개의 파일이 생성되었습니다:
  - 정제된 데이터: '식품데이터_최종_정제_ver4.json'
  - 삭제된 데이터: '식품데이터_삭제됨_ver4.json'


In [12]:
import json

def load_json_data(filepath):
    """JSON 파일을 읽어와 파이썬 객체로 변환하는 함수"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"오류: '{filepath}' 파일을 찾을 수 없습니다.")
        return None
    except json.JSONDecodeError:
        print(f"오류: '{filepath}' 파일의 JSON 형식이 올바르지 않습니다.")
        return None

def save_json_data(filepath, data):
    """데이터를 JSON 파일로 저장하는 함수"""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def analyze_single_allergen(ingredient, rules):
    """
    하나의 원재료에 대해 사용자가 요청한 최종 우선순위를 적용하여 분석하는 함수.
    """
    
    # --- 1단계: 카테고리 전체 제외(Exclusion) 규칙 최우선 검사 ---
    for exclusion_word in rules.get("exclusions", []):
        if exclusion_word in ingredient:
            return None

    # --- 2단계: 위험(Danger) 규칙 검사 ---
    if "danger" in rules:
        danger_rule = rules["danger"]
        
        # 2-1. 'list' 키워드와의 완전 일치 확인
        if ingredient in danger_rule.get("list", []):
            return "danger"
        
        # 2-2. 'list' 키워드의 포함(substring) 여부 확인
        for keyword in danger_rule.get("list", []):
            if keyword in ingredient:
                return "danger"

        # 2-3. 'exact_match' 키워드와의 완전 일치 확인
        if ingredient in danger_rule.get("exact_match", []):
            return "danger"

        # 2-4. 접미사(Suffix) 규칙 확인
        for suffix in danger_rule.get("suffix", []):
            if ingredient.endswith(suffix):
                if ingredient not in danger_rule.get("suffix_exclusions", []):
                    return "danger"

        # 2-5. 접두사(Prefix) 규칙 확인
        for prefix in danger_rule.get("prefix", []):
            if ingredient.startswith(prefix):
                is_prefix_excluded = any(ingredient.startswith(ex) for ex in danger_rule.get("prefix_exclusions", []))
                if not is_prefix_excluded:
                    return "danger"

    # --- 3단계: 주의(Caution) 규칙 검사 ---
    if "caution" in rules:
        caution_rule = rules["caution"]
        
        # 3-1. 'list' 키워드와의 완전 일치 확인
        if ingredient in caution_rule.get("list", []):
            return "caution"
        
        # 3-2. 'list' 키워드의 포함(substring) 여부 확인
        for keyword in caution_rule.get("list", []):
            if keyword in ingredient:
                return "caution"

        # 3-3. 'exact_match' 키워드와의 완전 일치 확인
        if ingredient in caution_rule.get("exact_match", []):
            return "caution"
                
        # 3-4. 접미사(Suffix) 규칙 확인
        for suffix in caution_rule.get("suffix", []):
            if ingredient.endswith(suffix):
                if ingredient not in caution_rule.get("suffix_exclusions", []):
                    return "caution"
                
        # 3-5. 접두사(Prefix) 규칙 확인
        for prefix in caution_rule.get("prefix", []):
            if ingredient.startswith(prefix):
                is_prefix_excluded = any(ingredient.startswith(ex) for ex in caution_rule.get("prefix_exclusions", []))
                if not is_prefix_excluded:
                    return "caution"

    return None


def analyze_all_products(products_list, allergy_rules):
    """전체 제품 리스트에 대해 알레르기 분석을 수행하는 메인 함수"""
    results_list = []
    for product in products_list:
        danger_allergens = set()
        caution_allergens = set()

        if "원재료명" in product and isinstance(product["원재료명"], list):
            for ingredient in product["원재료명"]:
                for allergen, rules in allergy_rules.items():
                    result_level = analyze_single_allergen(ingredient, rules)
                    
                    if result_level == "danger":
                        danger_allergens.add(allergen)
                    elif result_level == "caution":
                        # 이 시점에서는 danger 여부와 상관없이 일단 추가합니다.
                        caution_allergens.add(allergen)

        final_caution_allergens = caution_allergens - danger_allergens

        product["allergy_analysis"] = {
            "danger": sorted(list(danger_allergens)),
            "caution": sorted(list(final_caution_allergens)) # 정리된 caution 목록을 사용
        }
        results_list.append(product)
    
    return results_list


# --- 스크립트 메인 실행 부분 ---
if __name__ == "__main__":
    allergy_rules_file = "../Data_Storage/표준화 데이터/Allergy_ver_3.json"
    products_file = "식품데이터_최종_표준화_ver4.json"
    # products_file = "product_test_data.json"
    output_file = "식품데이터_알레르기_분류_결과_ver_1.3.json"
    # output_file = "test_Allergy_data3.1.json"

    print("알레르기 규칙 및 제품 데이터를 로드합니다...")
    allergy_rules = load_json_data(allergy_rules_file)
    products_list = load_json_data(products_file)

    if allergy_rules and products_list:
        print("데이터 로드 완료. 알레르기 분석을 시작합니다...")
        
        final_results = analyze_all_products(products_list, allergy_rules)
        
        save_json_data(output_file, final_results)
        print(f"분석 완료! 결과가 '{output_file}' 파일에 저장되었습니다.")

알레르기 규칙 및 제품 데이터를 로드합니다...
데이터 로드 완료. 알레르기 분석을 시작합니다...
분석 완료! 결과가 '식품데이터_알레르기_분류_결과_ver_1.2.json' 파일에 저장되었습니다.
